In [62]:
import pandas as pd

df_test = pd.read_csv("test.csv")
df_test.columns
df_test["Fam"] = df_test["SibSp"] + df_test["Parch"]
df_test.drop(columns=["Ticket","Fare", "SibSp", "Parch"], inplace=True)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   Cabin        91 non-null     object 
 6   Embarked     418 non-null    object 
 7   Fam          418 non-null    int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 26.3+ KB


In [63]:
df_train = pd.read_csv("train.csv")
df_train["Fam"] = df_train["SibSp"] + df_train["Parch"]
df_train.drop(columns=["Ticket","Fare", "SibSp", "Parch"], inplace=True)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   Cabin        204 non-null    object 
 7   Embarked     889 non-null    object 
 8   Fam          891 non-null    int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 62.8+ KB


In [64]:
for col in ["Survived","Pclass","Sex","Embarked", "Fam"]:
    print()
    print(df_train[col].value_counts(dropna=False))



Survived
0    549
1    342
Name: count, dtype: int64

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Sex
male      577
female    314
Name: count, dtype: int64

Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

Fam
0     537
1     161
2     102
3      29
5      22
4      15
6      12
10      7
7       6
Name: count, dtype: int64


In [65]:
"""
NULL VALUES IN EMBARKED
"""
df_train["Embarked"]=df_train["Embarked"].fillna(value="S")

df_train["Embarked"].value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [66]:
"""
NULL VALUES IN AGE
"""

import numpy as np

titles = ["Mr.","Mrs.","Miss."]

def extract_title(name):
    for t in titles:
        if t in name:
            return t
    return np.nan

df_train["Title"] = df_train["Name"].map(extract_title)
df_test["Title"] = df_test["Name"].map(extract_title)

df_train["Title"] = df_train["Title"].fillna("Unknown")
df_test["Title"] = df_test["Title"].fillna("Unknown")

ages_mean_train = df_train.groupby("Title")["Age"].mean().round(1)
ages_mean_test = df_test.groupby("Title")["Age"].mean().round(1)

ages_mean_train['Unknown'] = df_train["Age"].mean().round(1)
ages_mean_test['Unknown'] = df_test["Age"].mean().round(1)
print(ages_mean_train)
print(ages_mean_test)

Title
Miss.      21.8
Mr.        32.4
Mrs.       35.9
Unknown    29.7
Name: Age, dtype: float64
Title
Miss.      21.8
Mr.        32.0
Mrs.       38.9
Unknown    30.3
Name: Age, dtype: float64


In [67]:
def fill_age_train(row):
    if pd.isna(row["Age"]):
        return ages_mean_train[row["Title"]]
    else:
        return row["Age"]
def fill_age_test(row):
    if pd.isna(row["Age"]):
        return ages_mean_train[row["Title"]]
    else:
        return row["Age"]
df_train["Age"] = df_train.apply(fill_age_train, axis=1)
df_test["Age"] = df_test.apply(fill_age_test, axis=1)

df_train[df_train["Age"].isna()]
df_test[df_test["Age"].isna()]

,PassengerId,Pclass,Name,Sex,Age,Cabin,Embarked,Fam,Title


In [68]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          418 non-null    float64
 5   Cabin        91 non-null     object 
 6   Embarked     418 non-null    object 
 7   Fam          418 non-null    int64  
 8   Title        418 non-null    object 
dtypes: float64(1), int64(3), object(5)
memory usage: 29.5+ KB


In [69]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   Cabin        204 non-null    object 
 7   Embarked     891 non-null    object 
 8   Fam          891 non-null    int64  
 9   Title        891 non-null    object 
dtypes: float64(1), int64(4), object(5)
memory usage: 69.7+ KB


In [70]:
"""
NULL VALUES IN CABIN
"""

df_train["Cabin"].unique()

array([nan, 'C85', 'C123', 'E46', 'G6', 'C103', 'D56', 'A6',
       'C23 C25 C27', 'B78', 'D33', 'B30', 'C52', 'B28', 'C83', 'F33',
       'F G73', 'E31', 'A5', 'D10 D12', 'D26', 'C110', 'B58 B60', 'E101',
       'F E69', 'D47', 'B86', 'F2', 'C2', 'E33', 'B19', 'A7', 'C49', 'F4',
       'A32', 'B4', 'B80', 'A31', 'D36', 'D15', 'C93', 'C78', 'D35',
       'C87', 'B77', 'E67', 'B94', 'C125', 'C99', 'C118', 'D7', 'A19',
       'B49', 'D', 'C22 C26', 'C106', 'C65', 'E36', 'C54',
       'B57 B59 B63 B66', 'C7', 'E34', 'C32', 'B18', 'C124', 'C91', 'E40',
       'T', 'C128', 'D37', 'B35', 'E50', 'C82', 'B96 B98', 'E10', 'E44',
       'A34', 'C104', 'C111', 'C92', 'E38', 'D21', 'E12', 'E63', 'A14',
       'B37', 'C30', 'D20', 'B79', 'E25', 'D46', 'B73', 'C95', 'B38',
       'B39', 'B22', 'C86', 'C70', 'A16', 'C101', 'C68', 'A10', 'E68',
       'B41', 'A20', 'D19', 'D50', 'D9', 'A23', 'B50', 'A26', 'D48',
       'E58', 'C126', 'B71', 'B51 B53 B55', 'D49', 'B5', 'B20', 'F G63',
       'C62 C64',

In [71]:
df_train["Cabin Amount"] = df_train["Cabin"].map(lambda x: 0 if pd.isna(x) else len(x.split()))
df_test["Cabin Amount"] = df_test["Cabin"].map(lambda x: 0 if pd.isna(x) else len(x.split()))

df_train["Deck"] = df_train["Cabin"].map(lambda x: "Unknown" if pd.isna(x) else x.split()[0][0])
df_test["Deck"] = df_test["Cabin"].map(lambda x: "Unknown" if pd.isna(x) else x.split()[0][0])

In [72]:
df_train = df_train.drop(columns=["Cabin", "Name"])
df_test= df_test.drop(columns=["Cabin", "Name"])

In [73]:
print(df_train.info())
print()
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   891 non-null    int64  
 1   Survived      891 non-null    int64  
 2   Pclass        891 non-null    int64  
 3   Sex           891 non-null    object 
 4   Age           891 non-null    float64
 5   Embarked      891 non-null    object 
 6   Fam           891 non-null    int64  
 7   Title         891 non-null    object 
 8   Cabin Amount  891 non-null    int64  
 9   Deck          891 non-null    object 
dtypes: float64(1), int64(5), object(4)
memory usage: 69.7+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   418 non-null    int64  
 1   Pclass        418 non-null    int64  
 2   Sex           418 non-null   

In [74]:
"""
NUMERIC ENCODING OF SEX
"""

df_train["Sex"] = df_train["Sex"].map({"male":0,"female":1})
df_test["Sex"] = df_test["Sex"].map({"male":0,"female":1})

In [75]:
"""
NUMERIC ENCODING OF DECK
"""

df_train.groupby("Deck")["Survived"].mean().sort_values(ascending=False)

Deck
D          0.757576
E          0.750000
B          0.744681
F          0.615385
C          0.593220
G          0.500000
A          0.466667
Unknown    0.299854
T          0.000000
Name: Survived, dtype: float64

In [76]:
deck_order = {
    "D":8, "E":7, "B":6, "F":5, "C":4, "G":3, "A":2, "Unknown":1, "T":0
}

df_train["Deck"] = df_train["Deck"].map(deck_order)
df_test["Deck"] = df_test["Deck"].map(deck_order)

In [77]:
"""
NUMERIC ENCODING OF TITLE
"""

df_train.groupby("Title")["Survived"].mean().sort_values(ascending=False)

Title
Mrs.       0.792000
Miss.      0.697802
Unknown    0.522388
Mr.        0.156673
Name: Survived, dtype: float64

In [78]:
df_train["Title"] = df_train["Title"].map({"Mrs.":3,"Miss.":2,"Unknown":1,"Mr.":0})
df_test["Title"] = df_test["Title"].map({"Mrs.":3,"Miss.":2,"Unknown":1,"Mr.":0})

In [79]:
"""
TURNING EMBARKED INTO DUMMIES
"""

df_train = pd.get_dummies(df_train, columns=["Embarked"], drop_first=True)
df_test = pd.get_dummies(df_test, columns=["Embarked"], drop_first=True)

In [80]:
print(df_train.info())
print()
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   891 non-null    int64  
 1   Survived      891 non-null    int64  
 2   Pclass        891 non-null    int64  
 3   Sex           891 non-null    int64  
 4   Age           891 non-null    float64
 5   Fam           891 non-null    int64  
 6   Title         891 non-null    int64  
 7   Cabin Amount  891 non-null    int64  
 8   Deck          891 non-null    int64  
 9   Embarked_Q    891 non-null    bool   
 10  Embarked_S    891 non-null    bool   
dtypes: bool(2), float64(1), int64(8)
memory usage: 64.5 KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   418 non-null    int64  
 1   Pclass        418 non-null    i

In [81]:
"""
MODEL TRAINING
"""
from sklearn.model_selection import train_test_split

X = df_train.drop(columns=["PassengerId","Survived"])
y = df_train["Survived"]

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=1)

In [88]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score


logreg = LogisticRegression(max_iter=100, random_state=1)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)

print("Validation Accuracy:", accuracy)
print("Confusion Matrix:\n", cm)


Validation Accuracy: 0.8044692737430168
Confusion Matrix:
 [[90 16]
 [19 54]]


In [89]:
"""
SUBMISSION
"""
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   418 non-null    int64  
 1   Pclass        418 non-null    int64  
 2   Sex           418 non-null    int64  
 3   Age           418 non-null    float64
 4   Fam           418 non-null    int64  
 5   Title         418 non-null    int64  
 6   Cabin Amount  418 non-null    int64  
 7   Deck          418 non-null    int64  
 8   Embarked_Q    418 non-null    bool   
 9   Embarked_S    418 non-null    bool   
dtypes: bool(2), float64(1), int64(7)
memory usage: 27.1 KB


In [90]:
X_test = df_test.drop(columns=["PassengerId"])
y_test_pred = logreg.predict(X_test)

submission = pd.DataFrame(
    {
        "PassengerId":df_test["PassengerId"],
        "Survived":y_test_pred
    }
)

submission.to_csv("submission.csv", index=False)